In [3]:
import requests
from tqdm import tqdm
import pandas as pd
import xarray as xr
from io import BytesIO
import ast
from datetime import date

In [4]:
URL = "https://downloads.psl.noaa.gov//Datasets/cpc_global_temp/tmax.{}.nc"
REGION_AREA_FILE = "../../data/temp/temp_region_area.csv"
OUTPUT_FILE = "../../data/temp/regional_temp_data.csv"

START_DATE = date(2018, 1, 1)
END_DATE = date(2025, 12, 31)
YEAR_RANGE = range(START_DATE.year,END_DATE.year+1)

In [ ]:
# this dataframe only contains coordinates whose 0.5x0.5 squares overlap with the US
df_region_area = pd.read_csv(REGION_AREA_FILE, index_col="coords", converters={"coords": ast.literal_eval})
regions = df_region_area.columns
df_region_area

,Region 1,Region 2,Region 3,Region 4,Region 5,Region 6,Region 7,Region 8,Region 9,Region 10
coords,,,,,,,,,,
"(-179.25, 51.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.018633
"(-179.25, 51.75)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.001345
"(-178.75, 51.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.017087
"(-178.75, 51.75)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.054108
"(-178.25, 28.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000962,0.000000
...,...,...,...,...,...,...,...,...,...,...
"(-65.75, 17.75)",0.0,0.010819,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
"(-65.75, 18.25)",0.0,0.507078,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
"(-65.25, 18.25)",0.0,0.053583,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [4]:
df = pd.DataFrame()
for year in tqdm(YEAR_RANGE):
    # get unprocessed dataframe
    year_url = URL.format(year)
    bytes = BytesIO(requests.get(year_url).content)
    ds = xr.open_dataset(bytes)
    df_year = ds.to_dataframe().dropna().reset_index()

    df_year["coords"] = list(zip(df_year["lon"]-360, df_year["lat"])) # lon must be adjusted to fit the shapefile's standard
    df_year_us = df_year.merge(df_region_area, on = "coords") # should result in 4633 unique US coords

    unique_days = df_year_us.time.unique()
    n_days = len(unique_days) # 365 or 366

    # we calculate the daily temperature per region by averaging the temperatures in each of the 4633 squares,
    # weighted by area of the square in the region as given by df_region_area
    # this is done using matrix multiplication
    x = df_year_us["tmax"].values.reshape(-1, 1, n_days).transpose(2, 1, 0) # 365/366 x 1 x 4633
    y = df_year_us[regions].values.reshape(-1, n_days, 10).transpose(1, 0, 2) # 365/366 x 4633 x 10
    normalizing_factor = y[0].sum(axis = 0) # after multiplying the matrices we divide by the total area per region

    df_year_processed = pd.DataFrame(
        (x @ y).squeeze(1) / normalizing_factor, # multiplication results in 365/366 x 1 x 10 (squeezed to 365/366 x 10)
        index = unique_days,
        columns = regions
    )

    # add to rest of years
    df = pd.concat([df, df_year_processed])

# filter to be between start and end dates
df = df[(df.index.date >= START_DATE) & (df.index.date <= END_DATE)]
df

100%|██████████| 8/8 [06:14<00:00, 46.83s/it]


,Region 1,Region 2,Region 3,Region 4,Region 5,Region 6,Region 7,Region 8,Region 9,Region 10
2018-01-01,-15.837069,-10.347318,-7.526191,0.256565,-15.160282,-0.363642,-13.126441,-9.710529,15.798783,-6.738495
2018-01-02,-12.189956,-5.388370,-5.612809,1.883143,-10.613858,1.298075,-4.330759,-1.674502,14.343217,-4.513651
2018-01-03,-5.170511,-3.484295,-2.236547,4.067688,-9.743023,9.192431,-1.263505,-1.196697,14.299780,-4.316342
2018-01-04,-4.535368,-5.003550,-4.980320,2.645091,-12.728201,10.064703,-1.880859,-0.142819,15.886730,-5.315687
2018-01-05,-6.707461,-11.330217,-9.039630,4.140429,-13.790228,13.971358,-2.651693,0.232164,16.154701,-5.215090
...,...,...,...,...,...,...,...,...,...,...
2025-12-27,-7.143334,-2.834758,7.618850,22.917213,5.218045,24.682628,16.362055,5.967286,9.848556,-17.823341
2025-12-28,-0.999950,3.444298,10.161617,21.810269,8.003743,21.329236,11.046786,-5.287031,9.045837,-18.548098
2025-12-29,3.223324,7.403524,13.728554,16.874758,-3.850697,8.679618,-2.133836,0.504085,10.344066,-16.120822
2025-12-30,-2.502194,-3.746495,-1.767334,6.743017,-3.643315,12.168838,7.988128,5.614722,10.514403,-15.098077


In [5]:
df.to_csv(OUTPUT_FILE)